# 01 — International location audit (NO outcome analysis)

**Scope decision:** the challenge's primary evaluation remains U.S.-focused. This is a separate, optional international research extension. We use the existing **globally cleaned** file (not the raw CSV), and only locations that explicitly end in a recognized non-U.S. country or region name. This is an *evidence-based location proxy*, **not verified Kickstarter account country**.

- `Berlin, Germany`, `Taipei, Taiwan`, `Toronto, Canada` → eligible explicit named region.
- `Chicago, IL` → existing U.S. state-code candidate, **not** international.
- `Portland, USA` → explicit U.S., **not** international.
- `Toronto, ON`, `Berlin, BE`, missing or malformed → **unknown**; DO NOT relabel these as international.
- U.S. territories are kept separate and excluded, as are ambiguous names like `Georgia`.

This notebook audits **location strings only**. Success/failure comparisons are delayed until `04`, using training data exclusively.

In [1]:
from pathlib import Path
import sys
HERE = Path.cwd()
PROJECT_ROOT = next((folder for folder in (HERE, *HERE.parents)
                     if (folder / "utils" / "preprocessing.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the 1C repository")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
GLOBAL_DIR = PROJECT_ROOT / "data" / "Processed-Data"
INT_DIR = GLOBAL_DIR / "international_only"
CLEAN_GLOBAL = GLOBAL_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv"
import pandas as pd
from utils.international_scope import location_scope_audit
from utils.us_scope import us_state_proxy_mask
if not CLEAN_GLOBAL.exists():
    raise FileNotFoundError(f"Missing {CLEAN_GLOBAL}; restore the global processed CSV (or run global 02)")
clean = pd.read_csv(CLEAN_GLOBAL)
assert clean["project id"].is_unique
classes, overview, countries, uncertain = location_scope_audit(clean)
print(f"Globally cleaned projects: {len(clean):,}")
print(f"Explicit non-U.S. named-location candidates: {classes['scope'].eq('explicit_non_us').sum():,}")
print(f"U.S. state/DC proxy candidates: {classes['scope'].eq('us_candidate').sum():,}")
print(f"Unknown/ambiguous location projects: {classes['scope'].eq('unknown').sum():,}")
display(overview)

Globally cleaned projects: 45,815
Explicit non-U.S. named-location candidates: 2,279
U.S. state/DC proxy candidates: 42,175
Unknown/ambiguous location projects: 1,333


,scope,evidence,n
0,explicit_non_us,explicit_country_or_region_name,2279
1,explicit_us,explicit_us_name,2
2,unknown,ambiguous_country_name,8
3,unknown,missing_location,1318
4,unknown,no_explicit_country_name,7
5,us_candidate,us_state_suffix,42175
6,us_territory,us_territory_excluded,26


In [2]:
print("Most represented explicitly named non-U.S. regions (all statuses):")
display(countries.head(30))
print("Top unknown location strings (review, not automatically included):")
display(uncertain.head(30))
assert not (classes["scope"].eq("explicit_non_us") & us_state_proxy_mask(clean)).any()
print("Verified the international-name set and U.S. state-suffix candidates do not overlap")

Most represented explicitly named non-U.S. regions (all statuses):


,country_region,n
0,Canada,284
1,United Kingdom,216
2,Germany,119
3,Mexico,84
4,France,81
5,Italy,74
6,India,68
7,China,60
8,Australia,57
9,Japan,55


Top unknown location strings (review, not automatically included):


,location,evidence,n
10,NaN,missing_location,1318
4,"Ciudad Aut���_noma De Buenos Aires, Argent",no_explicit_country_name,4
9,"Tbilisi, Georgia",ambiguous_country_name,3
0,"Batumi, Georgia",ambiguous_country_name,1
1,"Brazzaville, Congo",ambiguous_country_name,1
2,"Bukavu, Congo",ambiguous_country_name,1
3,"Charlottesville, Virginia",no_explicit_country_name,1
5,"Kinshasa, Congo",ambiguous_country_name,1
6,Maar,no_explicit_country_name,1
7,"Santiago De Los Caballeros, Dominican Re",no_explicit_country_name,1


Verified the international-name set and U.S. state-suffix candidates do not overlap


## Audit conclusion and decision gate

Every assigned country is derived **only from the final literal component of `location`**, with conservative exact-name matching (including common aliases). Names and numeric counts are *provisional* until reviewed against original Kaggle/source information. Country/region labels are linguistic groupings and do not establish nationality, currency, creator residence, or political status. Do not infer country from a city alone; do not treat unknown locations as a separate international population.
